In [ ]:
#instract text from a document
!pip install pymupdf
#text to embeddings using sentence transformers
!pip install -U sentence-transformers
#call Groq model
!pip install groq

In [ ]:
from IPython.core.interactiveshell import no_op
#pdf
import fitz
#numerical
import numpy as np
import os
#text to embeddings
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata


In [ ]:
#Getting Groq API Key
GROQ_api_key = userdata.get('groq_API_KEY')
groq_client = Groq(api_key=GROQ_api_key)
#get embinding from hugging face
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
def extract_and_chunk_pdf(pdf_path, chunk_size=500, overlap=100):
    """
    Extracts text page-by-page. Automatically falls back to Tesseract OCR
    if a page contains only images or scanned data.
    """
    print(f"Opening {pdf_path}...")
    doc = fitz.open(pdf_path)
    full_text = ""

    for page_num, page in enumerate(doc):
        # Attempt standard text extraction first
        text = page.get_text("text").strip()

        # If the page has very little or no selectable text, it's likely an image/scan
        if len(text) < 50:
            print(f" -> Page {page_num + 1} appears to be an image. Running OCR...")
            try:
                # Tell PyMuPDF to extract text by rendering the page through its OCR engine
                ocr_textpage = page.get_textpage_ocr(flags=3, language="eng")
                text = page.get_text("text", textpage=ocr_textpage)
            except Exception as e:
                print(f"OCR failed on page {page_num + 1}: {e}")
                text = ""
        else:
            print(f" -> Page {page_num + 1} extracted via standard digital text.")

        full_text += text + "\n"

    # Structural word-based chunking logic (same as before)
    words = full_text.split()
    chunks = []
    word_chunk_size = chunk_size // 5
    word_overlap = overlap // 5

    for i in range(0, len(words), word_chunk_size - word_overlap):
        chunk_words = words[i:i + word_chunk_size]
        chunk_text = " ".join(chunk_words)
        if len(chunk_text.strip()) > 10:
            chunks.append(chunk_text)

    print(f"\nTotal Pipeline Ready: Created {len(chunks)} text chunks.")
    return chunks


In [ ]:
# #chunking creating a funciton that can process a pdf
# def extract_and_chunk_pdf(pdf_path, chunk_size=500, overlap=100):
#   print("extracting pdf")
#   #open the pdf
#   doc = fitz.open(pdf_path)
#   # 2. Extract text
#   full_text = ""

#   for page in doc:
#       full_text += page.get_text("text") + "\n"

#   # 3. Turn text into words
#   words = full_text.split()

#   # 4. Convert character-ish settings to word settings
#   word_chunk_size = chunk_size // 5
#   word_overlap = overlap // 5

#   # 5. Create overlapping chunks
#   chunks = []

#   for i in range(0, len(words), word_chunk_size - word_overlap):
#       chunk_words = words[i:i + word_chunk_size]
#       chunk_text = " ".join(chunk_words)

#       if len(chunk_text.strip()) > 10:
#           chunks.append(chunk_text)

#   print(f"Created {len(chunks)} overlapping text chunks.")

#   return chunks

In [ ]:
#vector database
 #3. VECTOR STORAGE & RETRIEVAL (FROM SCRATCH)
class SimpleVectorDB:
    """An educational in-memory vector database using raw NumPy math."""
    def __init__(self):
        self.chunks = []
        self.embeddings = []

    def add_documents(self, text_chunks, embedding_model):
        self.chunks.extend(text_chunks)
        # Compute embeddings for all text segments
        vectors = embedding_model.encode(text_chunks, convert_to_numpy=True)
        if len(self.embeddings) == 0:
            self.embeddings = vectors
        else:
            self.embeddings = np.vstack((self.embeddings, vectors))

    def search(self, query, embedding_model, top_k=3):
        # Embed the incoming user query
        query_vector = embedding_model.encode([query], convert_to_numpy=True)[0]

        # Calculate Cosine Similarity mathematically: (A dot B) / (||A|| * ||B||)
        dot_products = np.dot(self.embeddings, query_vector)
        norm_db = np.linalg.norm(self.embeddings, axis=1)
        norm_query = np.linalg.norm(query_vector)

        similarities = dot_products / (norm_db * norm_query)

        # Sort results from highest match score to lowest
        top_indices = np.argsort(similarities)[::-1][:top_k]

        return [self.chunks[idx] for idx in top_indices]

In [ ]:
#excute
# 1. Process your uploaded PDF file
pdf_filename = "catch_framwork.pdf"  # Match this with your file name in Colab

if not os.path.exists(pdf_filename):
    raise FileNotFoundError(f"Please upload a PDF named '{pdf_filename}' into your Colab files panel.")

text_chunks = extract_and_chunk_pdf(pdf_filename)

# 2. Index the text chunks inside our custom Vector DB
vector_db = SimpleVectorDB()
print("Generating embeddings and building vector index...")
vector_db.add_documents(text_chunks, embedding_model)
print("Pipeline is ready!")


Opening catch_framwork.pdf...
 -> Page 1 appears to be an image. Running OCR...
 -> Page 2 appears to be an image. Running OCR...
 -> Page 3 appears to be an image. Running OCR...
 -> Page 4 appears to be an image. Running OCR...
 -> Page 5 appears to be an image. Running OCR...
 -> Page 6 appears to be an image. Running OCR...
 -> Page 7 appears to be an image. Running OCR...

Total Pipeline Ready: Created 2 text chunks.
Generating embeddings and building vector index...
Pipeline is ready!


In [ ]:
# Execute
# 1. Process your uploaded PDF file
pdf_filename = "catch_framwork.pdf"

if not os.path.exists(pdf_filename):
    raise FileNotFoundError(
        f"Please upload a PDF named '{pdf_filename}' into your Colab files panel."
    )

text_chunks = extract_and_chunk_pdf(pdf_filename)

# 2. Index the text chunks inside our custom Vector DB
vector_db = SimpleVectorDB()

print("Generating embeddings and building vector index...")
vector_db.add_documents(text_chunks, embedding_model)

print("Pipeline is ready!")

print("Number of text chunks:", len(text_chunks))
print("Number of embeddings:", len(vector_db.embeddings))

Opening catch_framwork.pdf...
 -> Page 1 appears to be an image. Running OCR...
 -> Page 2 appears to be an image. Running OCR...
 -> Page 3 appears to be an image. Running OCR...
 -> Page 4 appears to be an image. Running OCR...
 -> Page 5 appears to be an image. Running OCR...
 -> Page 6 appears to be an image. Running OCR...
 -> Page 7 appears to be an image. Running OCR...

Total Pipeline Ready: Created 2 text chunks.
Generating embeddings and building vector index...
Pipeline is ready!
Number of text chunks: 2
Number of embeddings: 2


In [ ]:
import numpy as np
#rag generation
 #5. GENERATION FUNCTION (RAG)
# =====================================================================
def ask_pdf(query):
    # Check if the vector database is empty
    if len(vector_db.embeddings) == 0:
        return "Error: The document database is empty. Please ensure the PDF was successfully processed and chunks were created."

    # Phase 1: Retrieve context
    retrieved_context = vector_db.search(query, embedding_model, top_k=3)

    # Handle cases where no context is retrieved (e.g., if search returns empty)
    if not retrieved_context:
        context_str = "No relevant context found in the document."
    else:
        context_str = "\n\n---\n\n".join(retrieved_context)

    # Phase 2: Construct the strict system payload prompt
        system_prompt = (
            "You are a hospitality training AI assistant helping a student respond to guests in simulated scenarios. "
    "Use the retrieved documents to guide your response using the CATCH framework: Care, Adaptability, Think, "
    "Create Exceptional Experiences, and Human Connection. "
    "Use the guest's situation and the CATCH principles to provide practical, specific actions the student should take. "
    "You may use general hospitality reasoning to recommend appropriate actions, but do not invent specific hotel policies, "
    "procedures, services, prices, timeframes, amenities, or other factual details that are not provided in the retrieved documents. "
    "Focus on what the student can reasonably say or do to address the guest's needs, show empathy, take ownership, "
    "adapt to the situation, and create an exceptional experience. "
    "If a specific fact or hotel procedure is required to answer the question and it is not available in the retrieved documents, "
    "say \"Not enough information\" for that specific fact rather than making it up. "
    "After your reasoning, always provide a concise final summary under the heading \"Recommendation:\" "
    "that gives the specific actions the student should take. "
    f"\n\nFACTUAL CONTEXT:\n{context_str}"
            #PROMBLEM: not giving summary
    #          "You are a hospitality training AI assistant helping a student respond to guests in simulated scenarios. "
    # "Use the retrieved documents to guide your response using the CATCH framework: Care, Adaptability, Think, "
    # "Create Exceptional Experiences, and Human Connection. "
    # "Use the guest's situation and the CATCH principles to provide practical, specific actions the student should take. "
    # "You may use general hospitality reasoning to recommend appropriate actions, but do not invent specific hotel policies, "
    # "procedures, services, prices, timeframes, amenities, or other factual details that are not provided in the retrieved documents. "
    # "Focus on what the student can reasonably say or do to address the guest's needs, show empathy, take ownership, "
    # "adapt to the situation, and create an exceptional experience. "
    # "If a specific fact or hotel procedure is required to answer the question and it is not available in the retrieved documents, "
    # "say \"Not enough information\" for that specific fact rather than making it up. "
    # f"\n\nFACTUAL CONTEXT:\n{context_str}"

    #          "You are a hospitality training AI assistant helping a student respond to guests in simulated scenarios. "
    # "Use the retrieved documents as your only source of factual information. "
    # "Apply the CATCH framework: Care, Adaptability, Think, Create Exceptional Experiences, and Human Connection. "
    # "Generate a natural, professional, empathetic response the student could give to the guest. "
    # "Adapt the response to the guest's specific situation and look for opportunities to create an exceptional experience. "
    # "Never invent or assume information. "
    # "If the retrieved documents do not contain enough information to answer, respond: \"Not enough information.\" "
    # "Then state the appropriate department, person, or location the guest should contact or visit, "
    # "only if that information is provided in the retrieved documents. "
    # f"\n\nFACTUAL CONTEXT:\n{context_str}"

            # "You are an expert hospitality assistant. Answer the user's question using ONLY "
            # "the provided factual context.\n\n"
            # "Use the CATCH framework internally to decide what the student should do. "
            # "Do not explain or reveal this process.\n\n"
            # "Return ONLY a concise, practical recommendation describing the specific actions "
            # "the student should take. The answer should tell the student what to DO to resolve "
            # "the guest's situation.\n\n" "Do NOT include <think> tags, reasoning, analysis, framework explanations, "
            # "CATCH categories, summaries of the situation, or explanations of how you reached "
            # "the recommendation. Do not discuss the quality or clarity of the provided context.\n\n"
            # "Keep the recommendation concise and actionable, normally 2–4 sentences.\n\n" f"FACTUAL CONTEXT:\n{context_str}"

          #PROBLEM: Not answer privided question too restrictive to document
          #  "You are an expert hospitality assistant. Based ONLY on the provided context, "
          #  "give the student a practical recommendation for what they should do next. "
          #  "Use the CATCH framework in the context to determine which aspect of CATCH "
          #  "the student most needs to apply based on what the guest has revealed. "
          #  "Then translate that into specific, actionable steps for resolving the guest's situation. "
          #  "Do not analyze the situation, explain your reasoning, describe the CATCH framework, "
          #  "or repeat the guest's statements. Do not simply identify a CATCH category. "
          #  "The response must focus on what the student should actually do. "
          #  "If multiple CATCH aspects are relevant, prioritize the one most important to the situation "
          #  "and incorporate the others only when they directly support the recommended actions. "
          #  "If the context does not provide enough information to make a recommendation, "
          #  "say 'I cannot find enough information in the document to make a recommendation.'\n\n" f"FACTUAL CONTEXT:\n{context_str}"


    )

    # Phase 3: Execute generation using a fast open-source model (Llama 3) via Groq
    completion = groq_client.chat.completions.create(
        model="qwen/qwen3.6-27b",  # Updated to a supported model string
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0,  # Kept at 0 to avoid creative hallucinations
        max_tokens=800
    )

    return completion.choices[0].message.content


# =====================================================================
# 6. TEST YOUR PIPELINE
# =====================================================================
# Modify this test string with a query relevant to your specific PDF content
test_query = "What is the CATCHE framework and how should it guide an AI assistant in a hospitality guest interaction?"
print(f"\nUser Query: {test_query}\n")

response = ask_pdf(test_query)
print(f"AI Response:\n{response}")


User Query: What is the CATCHE framework and how should it guide an AI assistant in a hospitality guest interaction?

AI Response:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "What is the CATCHE framework and how should it guide an AI assistant in a hospitality guest interaction?"
   - **Note:** The user wrote "CATCHE" but the prompt and documents clearly refer to "CATCH". I should gently correct/clarify this while answering.
   - **Key Elements to Address:**
     - Define the CATCH framework based on the provided documents.
     - Explain how it should guide an AI assistant in hospitality guest interactions.
     - Use the CATCH principles: Care, Adaptability, Think, Create Exceptional Experiences, Human Connection.
     - Align with the prompt's instructions (focus on practical guidance, use retrieved docs, avoid inventing facts, provide a "Recommendation:" section).

2.  **Extract Information from Retrieved Documents:**
   - The document is a

In [ ]:
test_query = "According to the CACHE framework, what does Human Connection mean in a hospitality guest interaction?"
print(f"\nUser Query: {test_query}\n")

response = ask_pdf(test_query)
print(f"AI Response:\n{response}")


User Query: According to the CACHE framework, what does Human Connection mean in a hospitality guest interaction?

AI Response:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "According to the CACHE framework, what does Human Connection mean in a hospitality guest interaction?"
   - **Note:** The user wrote "CACHE" but the prompt and retrieved documents clearly refer to "CATCH". I should gently correct/clarify this while answering based on the provided text.
   - **Key Concept:** "Human Connection" within the CATCH framework.

2.  **Scan Retrieved Documents for Keywords:**
   - Keywords: "CATCH", "Human Connection", "H", "Human"
   - The retrieved text is quite garbled/OCR-scanned, but I can see:
     - "CATCH Human Capability Framework © Yaw"
     - "Understanding what matters to others — and responding with judgment... empathy..."
     - "Adjusting intelligently when conditions change." (Adaptability)
     - "Framing the right problem and... maki

In [ ]:
test_query = """
A hotel guest is upset because their room was not properly serviced.
What should the AI assistant consider when responding to this situation using the CACHE framework?
"""

print(f"\nUser Query: {test_query}\n")

response = ask_pdf(test_query)
print(f"AI Response:\n{response}")


User Query: 
A hotel guest is upset because their room was not properly serviced.
What should the AI assistant consider when responding to this situation using the CACHE framework?


AI Response:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Scenario:** A hotel guest is upset because their room was not properly serviced.
   - **Question:** What should the AI assistant consider when responding to this situation using the CATCH framework? (Note: User wrote "CACHE" but clearly means "CATCH" based on the prompt and context).
   - **Framework:** CATCH (Care, Adaptability, Think, Create Exceptional Experiences, Human Connection)
   - **Constraints:** Use retrieved documents (provided in FACTUAL CONTEXT), guide response using CATCH, provide practical/specific actions, do not invent policies/prices/amenities, say "Not enough information" if specific facts are missing, end with "Recommendation:" summary.

2.  **Analyze Retrieved Documents (FACTUAL CONTEXT):**
   - The

In [ ]:
test_query = "“The guest has told you: ‘It’s been a long day. I was just hoping to come back and not have to think about anything.’ The room was not serviced properly, and the guest originally asked only for fresh towels. Based on everything the guest has revealed so far, what specific actions should the student take to resolve the situation? Give a practical recommendation, not an analysis.”"
print(f"\nUser Query: {test_query}\n")

response = ask_pdf(test_query)
print(f"AI Response:\n{response}")


User Query: “The guest has told you: ‘It’s been a long day. I was just hoping to come back and not have to think about anything.’ The room was not serviced properly, and the guest originally asked only for fresh towels. Based on everything the guest has revealed so far, what specific actions should the student take to resolve the situation? Give a practical recommendation, not an analysis.”

AI Response:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Guest Statement:** "It’s been a long day. I was just hoping to come back and not have to think about anything."
   - **Situation:** Room was not serviced properly. Guest originally asked only for fresh towels.
   - **Task:** Provide specific actions the student should take to resolve the situation based on the guest's revelation and the CATCH framework. Give a practical recommendation, not an analysis.
   - **Constraints:** Use CATCH framework (Care, Adaptability, Think, Create Exceptional Experiences, Human Conne

In [ ]:
test_query = "“The guest has told you: ‘It’s been a long day. I was just hoping to come back and not have to think about anything.’ The room was not serviced properly, and the guest originally asked only for fresh towels. Based on everything the guest has revealed so far, what specific actions should the student take to resolve the situation? Give a practical summarized recommendation”"
print(f"\nUser Query: {test_query}\n")

response = ask_pdf(test_query)
print(f"AI Response:\n{response}")


User Query: “The guest has told you: ‘It’s been a long day. I was just hoping to come back and not have to think about anything.’ The room was not serviced properly, and the guest originally asked only for fresh towels. Based on everything the guest has revealed so far, what specific actions should the student take to resolve the situation? Give a practical summarized recommendation”

AI Response:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Guest Statement:** "It’s been a long day. I was just hoping to come back and not have to think about anything."
   - **Situation:** Room was not serviced properly. Guest originally only asked for fresh towels.
   - **Task:** Provide specific actions the student should take to resolve the situation, based on the guest's statement and the CATCH framework. Give a practical summarized recommendation.
   - **Constraints:** Use CATCH framework (Care, Adaptability, Think, Create Exceptional Experiences, Human Connection). Use r

In [ ]:
test_query = "“The guest says: ‘I know check-in isn't until later, but I've just arrived after a long flight and I'm exhausted. Is there anything you can do to help me? I don't really want to sit around waiting.’ What should the student do?”"
print(f"\nUser Query: {test_query}\n")

response = ask_pdf(test_query)
print(f"AI Response:\n{response}")


User Query: “The guest says: ‘I know check-in isn't until later, but I've just arrived after a long flight and I'm exhausted. Is there anything you can do to help me? I don't really want to sit around waiting.’ What should the student do?”

AI Response:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Guest Statement:** "I know check-in isn't until later, but I've just arrived after a long flight and I'm exhausted. Is there anything you can do to help me? I don't really want to sit around waiting."
   - **Question:** What should the student do?
   - **Context:** Hospitality training AI assistant using the CATCH framework (Care, Adaptability, Think, Create Exceptional Experiences, Human Connection).
   - **Constraints:** Use retrieved documents (which are mostly garbled/OCR errors but contain the CATCH framework names: Care, Adaptability, Think, Create Exceptional Experiences, Human Connection). Do not invent specific hotel policies, procedures, services, prices,

Code with recommendation only

In [ ]:
import numpy as np
#rag generation
 #5. GENERATION FUNCTION (RAG)
# =====================================================================
def ask_pdf(query):
    # Check if the vector database is empty
    if len(vector_db.embeddings) == 0:
        return "Error: The document database is empty. Please ensure the PDF was successfully processed and chunks were created."

    # Phase 1: Retrieve context
    retrieved_context = vector_db.search(query, embedding_model, top_k=3)

    # Handle cases where no context is retrieved (e.g., if search returns empty)
    if not retrieved_context:
        context_str = "No relevant context found in the document."
    else:
        context_str = "\n\n---\n\n".join(retrieved_context)

    # Phase 2: Construct the strict system payload prompt

    recommendation_system_prompt = (
    "You are a hospitality training assistant. "
    "Based on the guest's situation and the provided context, give the student one clear, practical recommendation "
    "for how they should respond to or handle the guest. "
    "Apply the CATCH framework: Care, Adaptability, Think, Create Exceptional Experiences, and Human Connection. "
    "Use general hospitality reasoning when appropriate, but do not invent hotel-specific policies, services, prices, "
    "timeframes, or other factual information that is not provided in the context. "
    "Do not provide analysis, reasoning, explanations, or a breakdown of the CATCH framework. "
    "Return ONLY the practical recommendation that the student should follow. "
    "Keep the recommendation concise, specific, professional, and actionable."
    f"\n\nCONTEXT:\n{context_str}"


    )

    # Phase 3: Execute generation using a fast open-source model (Llama 3) via Groq
    completion = groq_client.chat.completions.create(
        model="qwen/qwen3.6-27b",  # Updated to a supported model string
        messages=[
            {"role": "system", "content": recommendation_system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0,  # Kept at 0 to avoid creative hallucinations
        max_tokens=500
    )

    return completion.choices[0].message.content


# =====================================================================
# 6. TEST YOUR PIPELINE
# =====================================================================
# Modify this test string with a query relevant to your specific PDF content
test_query = "What is the CATCHE framework and how should it guide an AI assistant in a hospitality guest interaction?"
print(f"\nUser Query: {test_query}\n")

response = ask_pdf(test_query)
print(f"AI Response:\n{response}")


User Query: What is the CATCHE framework and how should it guide an AI assistant in a hospitality guest interaction?

AI Response:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "What is the CATCHE framework and how should it guide an AI assistant in a hospitality guest interaction?"
   - **Context Provided:** The context is a garbled/OCR-scanned text about the "CATCH Human Capability Framework" by Yaw. It mentions:
     - C: Care (Understanding what matters to others, responding with empathy, etc.)
     - A: Adaptability (Adjusting intelligently when conditions change)
     - T: Think (Framing the right problem, making defensible judgments, owning consequences)
     - C: Create Exceptional Experiences
     - H: Human Connection
   - **System Prompt Constraints:**
     - Role: Hospitality training assistant
     - Task: Give ONE clear, practical recommendation for how the student should respond to/handle the guest based on the situation and context

In [ ]:
test_query = "“The guest has told you: ‘It’s been a long day. I was just hoping to come back and not have to think about anything.’ The room was not serviced properly, and the guest originally asked only for fresh towels. Based on everything the guest has revealed so far, what specific actions should the student take to resolve the situation? Give a practical summarized recommendation”"
print(f"\nUser Query: {test_query}\n")

response = ask_pdf(test_query)
print(f"AI Recommendation:\n{response}")


User Query: “The guest has told you: ‘It’s been a long day. I was just hoping to come back and not have to think about anything.’ The room was not serviced properly, and the guest originally asked only for fresh towels. Based on everything the guest has revealed so far, what specific actions should the student take to resolve the situation? Give a practical summarized recommendation”

AI Recommendation:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Guest Statement:** “It’s been a long day. I was just hoping to come back and not have to think about anything.”
   - **Situation:** Room was not serviced properly. Guest originally asked only for fresh towels.
   - **Task:** Provide a specific, practical recommendation for the student to resolve the situation.
   - **Framework:** CATCH (Care, Adaptability, Think, Create Exceptional Experiences, Human Connection) - must be applied implicitly in the recommendation.
   - **Constraints:** 
     - One clear, practical r